* Calculate electron density from the PDB using DENSS: denss.pdb2mrc.py -f input.pdb -r 10 output -s 461.86 -v 1.00 and do not forget to multiply the resolution in DENSS with a sigma(resolution) factor to make sure it matches the molmap command in ChimeraX and EMAN2. 

In [ ]:
import condor
import tqdm

import os

import numpy as np

import scipy.fft as fft

import scipy.constants as constants
from fractions import Fraction

from skimage.measure import block_reduce

import matplotlib.pyplot as plt
import matplotlib.colors

from matplotlib.colors import LogNorm
from helper_functions import (write_text, radial, electron_density_to_dn, add_water_saxs)

import sys
from sys import stderr
from datetime import date
from IPython.display import clear_output
import time

import h5py

# Elementary constants
pi = constants.pi
e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

In [ ]:
showAutoCorr = True

# Loading normal AGIPD detector mask and crop it around beam center
bg_mask = 'emc/make_detector/agipd_detector_mask.h5'
with h5py.File(bg_mask, 'r') as det:
    det_mask = det['mask'][:]
write_text(f'Shape of detector file: {det_mask.shape}\n')
write_text('Cropping AGIPD detector to square array!\n')
cy, cx = det_mask.shape[0] // 2, det_mask.shape[1] // 2
det_mask = det_mask[cy-cx:cy+cx]
cy, cx = det_mask.shape[0] // 2, det_mask.shape[1] // 2

write_text(f'Shape of detector mask after cropping: {det_mask.shape}\n------------------------------------------------\n')

phot_eV = 9000 # beam energy in eV
phot_J = phot_eV * e # beam energy in J
phot_m = (h * c) / phot_J # beam wavelength in m 
### 50 uJ is in the paper measuring 14 nm diameter - more
pulse_energy = 200e-6 # pulse energy in J
beam_pol = 'horizontal' # beam polarization
beam_profile = 'gaussian' # beam profile

# Detector parameters
dsf = 4
if dsf == 1:
    pixel_size = 200e-6
    dimX = det_mask.shape[1]
    dimY = det_mask.shape[0]
else:
    pixel_size = dsf * 200e-6
    dimX = det_mask.shape[1] // dsf
    dimY = det_mask.shape[0] // dsf
pixel_num_x = dimX - dimX // 2
pixel_num_y = dimY - dimY // 2

det_dist = 0.4
D_particle = 15e-9 # assume largest size for GroEL

# Beam diameter
focus_diam = 14e-9
focus_rad = focus_diam / 2
write_text(f'Beam diameter: {focus_diam * 1e6} um\n')

# Beam area 
beam_area = pi * (focus_rad ** 2)
write_text(f'Beam area: {beam_area * 1e12} um\u00b2\n') 

# Beam photon count 
beam_phots = pulse_energy / phot_J
write_text(f'Photon count: {beam_phots} photons\n')

# Beam fluence 
beam_fluence = beam_phots / beam_area
write_text(f'Beam fluence: {beam_fluence * 1e-12:.2g} ph/um\u00b2\n')
write_text(f'Beam fluence: {beam_fluence * phot_J * 1e-6} uJ/um\u00b2\n------------------------------------------------\n')

# Edge resolution 
theta_pixel_x = 0.5 * np.arctan((pixel_num_x * pixel_size) / det_dist)
theta_pixel_y = 0.5 * np.arctan((pixel_num_y * pixel_size) / det_dist)

resolution_x = phot_m / (2.0 * np.sin(theta_pixel_x))
resolution_y = phot_m / (2.0 * np.sin(theta_pixel_y))
pix_real_x = 0.5 * resolution_x 
pix_real_y = 0.5 * resolution_y

# Maximum corner resolution 
center_to_corner = np.sqrt((pixel_num_x * pixel_size) ** 2 + (pixel_num_x * pixel_size) ** 2)
theta_max = 0.5 * np.arctan(center_to_corner / det_dist)
resolution_max = phot_m / (2.0 * np.sin(theta_max))

oversampling = (det_dist * phot_m) / (pixel_size * D_particle)
shannon_groel = 1 /( D_particle * 1e9)

write_text(f'Oversampling ratio: {oversampling}\n')
write_text(f'Detector pixel size: {pixel_size * 1e6} um\n')
write_text(f'Shannon pixel GroEL: {shannon_groel} nm^-1\n')
write_text(f'Corner resolution: {resolution_max * 1e9} nm\n')
write_text(f'Resolution along X-dimension: {resolution_x * 1e9} nm\n') 
write_text(f'Resolution along Y-dimension: {resolution_y * 1e9} nm\n')
write_text(f'Pixel size (real-space) along X-dimension: {pix_real_x * 1e9} nm\n')
write_text(f'Pixel size (real-space) along Y-dimension: {pix_real_y * 1e9} nm\n')

# Source 
source = condor.Source(wavelength=phot_m, pulse_energy=pulse_energy, focus_diameter=focus_diam, polarization=beam_pol, profile_model=beam_profile)

map3d, dx = condor.utils.emdio.read_map('denss/1ss8_denss.mrc')
map3d_scaled = electron_density_to_dn(map3d, phot_m)

# Particle map 
part_map = condor.ParticleMap(geometry='custom', map3d=map3d_scaled, dx=dx, rotation_formalism='random')

particle_set = {'particle_map' : part_map}

# Detector  
detector = condor.Detector(distance=det_dist, pixel_size=pixel_size, nx=dimX, ny=dimY)

# Experiment
condor_experiment = condor.Experiment(source, particle_set, detector)

In [ ]:
v_max = 1.0
cm = 'viridis'

if dsf == 1:
    det_mask_ds = det_mask
else:
    d_mask_float = det_mask.astype(float)
    d_mask_float[det_mask==False] = np.nan
    
    det_mask_ds = block_reduce(d_mask_float, block_size=dsf, func=np.nansum)
    write_text(f'{np.unique(det_mask_ds)}\n')
    
    plt.figure(dpi=100)
    plt.imshow(det_mask_ds, cmap='gray')
    plt.colorbar();
    
    det_mask_ds = (det_mask_ds >= 1.) # 56. for 12x downsampling, 15. for 7x downsampling, 2. for 6x downsampling, and 1. for 4x downsampling

plt.figure(dpi=100)
plt.imshow(det_mask_ds, cmap='gray')
plt.colorbar()
plt.plot(det_mask_ds.shape[1]//2, det_mask_ds.shape[0]//2, 'ro');

frac_good = det_mask_ds.sum()/(det_mask_ds.shape[0]*det_mask_ds.shape[1])
write_text(f'Fraction of good pixels : {frac_good}\n')

pat = np.ones_like(det_mask_ds)
water_bg = add_water_saxs(pat, pixel_size, det_dist, phot_m, pulse_energy)

<h3> SAXS water background </h3>

In [ ]:
rng = np.random.default_rng()
poiss_test = rng.poisson(lam=water_bg)

plt.figure(dpi=100)
plt.imshow(poiss_test,vmin=0, vmax=1.0,cmap=cm)
cb = plt.colorbar()
plt.title(f'number of photons: {poiss_test.sum()}', weight='bold')
plt.xticks([])
plt.yticks([]);

In [ ]:
sim_start, sim_end, sim_c = 0, 1, 1
n_sim = 9
det_mask_ds_stack = np.broadcast_to(det_mask_ds, (n_sim,) + det_mask_ds.shape).astype(np.float64)
water_stacked = np.broadcast_to(water_bg, (n_sim,) + water_bg.shape).astype(np.float64)

for s in range(sim_start,sim_end):
    print(f'Simulating round {sim_c}/{sim_end-sim_start}...')
    print(f'Simulating {n_sim} diffraction patterns...')
    particle_intens = np.zeros(shape=(n_sim, dimY, dimX))
    particle_real = np.zeros(shape=(n_sim, dimY, dimX))
    particle_orientations = np.zeros(shape=(n_sim, 4))
    
    # Resetting the RNG every simulation
    def_rng = np.random.default_rng()

    time_now = time.localtime(time.time())
    for i in tqdm.tqdm(np.arange(n_sim), colour='green'):
        result = condor_experiment.propagate()
        data_ampl = result['entry_1']['data_1']['data_fourier']
        real_space = np.fft.fftshift(np.fft.ifftn(data_ampl))
        I = np.abs(data_ampl) ** 2
        particle_intens[i, :, :] = I
        particle_real[i, :, :] = np.fft.fftshift(np.fft.ifftn(np.fft.fftshift(I)))
        particle_orientations[i] = result['particles']['particle_00']['extrinsic_quaternion']

    # Setting bad pixels in scattering pattern to 0.0
    particle_intens_masked = particle_intens.copy()
    particle_intens_water_masked = particle_intens_masked + water_stacked
    particle_intens_masked[det_mask_ds_stack==False] = 0.0
    particle_intens_water_masked[det_mask_ds_stack==False] = 0.0

    # Poisson sampling continuous protein scattering pattern
    poiss_samp = def_rng.poisson(lam=particle_intens_masked)
    poiss_samp_water = def_rng.poisson(lam=particle_intens_water_masked)
    
write_text('All simulations done...')

In [ ]:
visRes = True
if visRes:
    cc = 136
    cc_crop = cc
    
    max_num = 9
    nr = 3
    nc = 3
    shrink_bar = 0.55
    
    max_v_diff = 0.05

    int_colors = True
    
    rand_sel = def_rng.choice(n_sim,size=max_num, replace=False, shuffle=True)

    fig_handle = plt.figure(constrained_layout=True, dpi=200)
    fig_handle.patch.set_facecolor('white')
    spec_handle = fig_handle.add_gridspec(nrows=nr, ncols=nc)
    spec_handle.update(hspace=0.1, wspace=0.6)
    
    # Noiseless diffraction intensity
    for i in range(max_num):
        sel = rand_sel[i]

        sel_diff = particle_intens[sel]
        sel_diff = sel_diff[cc-cc_crop:cc+cc_crop,cc-cc_crop:cc+cc_crop]

        ax_i = fig_handle.add_subplot(spec_handle[i])
        im_i = plt.imshow(sel_diff,vmin=0.0,vmax=max_v_diff,cmap='viridis',interpolation=None)
        ax_i.set_xticks([])
        ax_i.set_yticks([])
        ax_i.set_title(f'{sel}',fontsize=7)
        ax_i.set_aspect('equal'); 

        if i==max_num-1: 
            minv, maxv = im_i.get_clim()
            c_bar_i = plt.colorbar(im_i,ax=ax_i,ticklocation = 'top',orientation='horizontal',shrink=shrink_bar) 
            c_bar_i.set_label('expectation value photon #', weight='bold',fontsize=5) 
            c_bar_i.set_ticks([minv,maxv])

    if showAutoCorr:
        # Autocorrelation
        max_v_corr = 0.1
        fig_handle = plt.figure(constrained_layout = True,dpi=200)
        fig_handle.patch.set_facecolor(f'white')
        spec_handle = fig_handle.add_gridspec(nrows = nr , ncols = nc) 
        spec_handle.update(hspace=0.1,wspace=0.6) 

        for i in range(max_num):
            sel = rand_sel[i]

            ax_i = fig_handle.add_subplot(spec_handle[i]) 
            im_i = plt.imshow(particle_real[sel], cmap='viridis', vmin=0, 
                              vmax=max_v_corr, interpolation=None) 
            ax_i.set_xticks([])
            ax_i.set_yticks([]) 
            ax_i.set_title(f'{sel}',fontsize=7) 
            ax_i.set_aspect('equal'); 
            if i==max_num-1: 
                minv, maxv = im_i.get_clim() 
                c_bar_i = plt.colorbar(im_i,ax=ax_i,ticklocation = 'top',orientation='horizontal',shrink=shrink_bar)
                c_bar_i.set_label('electron density in a.u.', weight='bold',fontsize=5) 
                c_bar_i.set_ticks([minv,maxv])

    # Poisson-sampled diffraction intensity 
    fig_handle = plt.figure(constrained_layout = True,dpi=200) 
    fig_handle.patch.set_facecolor(f'white') 

    spec_handle = fig_handle.add_gridspec(nrows = nr , ncols = nc) 
    spec_handle.update(hspace=0.1,wspace=0.6) 
    
    if int_colors:
        min_v = 0
        max_v = 2
        cm = plt.get_cmap('plasma', max_v+1)
    else:
        min_v = 0
        max_v = 0.01

    for i in range(max_num): 
        sel = rand_sel[i]
        sel_pat = poiss_samp[sel]

        sel_pat = sel_pat[cc-cc_crop:cc+cc_crop,cc-cc_crop:cc+cc_crop]
        
        ax_i = fig_handle.add_subplot(spec_handle[i]) 
        plt.suptitle(f'Poisson-only \n Mean photons/frame: {sel_pat.sum(axis=(0,1)).mean():.1f}\n')
        if int_colors:
            im_i = plt.imshow(sel_pat,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
        else:
            im_i = plt.imshow(sel_pat,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
        ax_i.set_xticks([])
        ax_i.set_yticks([])
        ax_i.set_title(f'{sel} - ({sel_pat.sum()} phs)',fontsize=7)
        ax_i.set_aspect('equal');
        if int_colors:
            if i==max_num-1: 
                minv, maxv = im_i.get_clim() 
                c_bar_i = plt.colorbar(im_i,ax=ax_i,ticklocation = 'top',orientation='horizontal',shrink=shrink_bar-0.2)
                c_bar_i.set_label('photon #', weight='bold',fontsize=5)
                c_bar_i.set_ticks(np.arange(min_v, max_v+1));
        else:
            if i==max_num-1: 
                plt.colorbar(im_i,ax=ax_i,ticklocation = 'top',orientation='horizontal',shrink=shrink_bar-0.2);
            
    # Poisson-sampled diffraction intensity with water
    fig_handle = plt.figure(constrained_layout = True,dpi=200) 
    fig_handle.patch.set_facecolor(f'white') 

    spec_handle = fig_handle.add_gridspec(nrows = nr , ncols = nc) 
    spec_handle.update(hspace=0.1,wspace=0.6) 

    if int_colors:
        min_v = 0
        max_v = 2
        cm = plt.get_cmap('plasma', max_v+1)
    else:
        min_v = 0
        max_v = 0.01

    for i in range(max_num): 
        sel = rand_sel[i]
        sel_pat = poiss_samp_water[sel]

        sel_pat = sel_pat[cc-cc_crop:cc+cc_crop,cc-cc_crop:cc+cc_crop]
            
        ax_i = fig_handle.add_subplot(spec_handle[i]) 
        plt.suptitle(f'Poisson + water \n Mean photons/frame: {sel_pat.sum(axis=(0,1)).mean():.1f}\n')
        if int_colors:
            im_i = plt.imshow(sel_pat,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
        else:
            im_i = plt.imshow(sel_pat,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
        ax_i.set_xticks([])
        ax_i.set_yticks([])
        ax_i.set_title(f'{sel} - ({sel_pat.sum()} phs)',fontsize=7) 
        ax_i.set_aspect('equal');
        if int_colors:
            if i==max_num-1: 
                minv, maxv = im_i.get_clim() 
                c_bar_i = plt.colorbar(im_i,ax=ax_i,ticklocation = 'top',orientation='horizontal',shrink=shrink_bar-0.2)
                c_bar_i.set_label('photon #', weight='bold',fontsize=5)
                c_bar_i.set_ticks(np.arange(min_v, max_v+1));
        else:
            if i==max_num-1:
                plt.colorbar(im_i,ax=ax_i,ticklocation = 'toujmp',orientation='horizontal',shrink=shrink_bar-0.2);

In [ ]:
# Additional downsampling for display in paper
sel = 0
dsf_additional = 3

sel_pat_poiss_only = poiss_samp[sel]
sel_pat_poiss_water = poiss_samp_water[sel]

sel_pat = sel_pat_poiss_water
sel_pat = sel_pat.astype(float)
sel_pat[det_mask_ds == False] = np.nan

plt.figure()
plt.imshow(sel_pat)
plt.colorbar();

sel_pat_ds_display = block_reduce(sel_pat, block_size=dsf_additional, func=np.nansum)

plt.figure()
plt.imshow(sel_pat_ds_display)
plt.colorbar();

In [ ]:
# Plotting code
plt.figure(dpi=140)
min_v = 0 
max_v = 4

cm = plt.get_cmap('inferno', max_v+1)

im_i = plt.imshow(sel_pat_ds_display,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
plt.xticks([]);
plt.yticks([]);
#c_bar = plt.colorbar(orientation='vertical', shrink=2.0) # for colorbar in panel figure --> shrink=2.0
#c_bar.set_ticks(np.arange(min_v, max_v+1));
#c_bar.ax.tick_params(labelsize=15);

#plt.savefig(f'prot_wat_pat_{sel}_dsf_{dsf_additional}.pdf',transparent=False, bbox_inches='tight');
#plt.savefig(f'cbar_dsf_{dsf_additional}.pdf',transparent=False, bbox_inches='tight'); # for colorbar
#plt.savefig(f'pat_{sel}_dsf_{dsf_additional*dsf}.pdf',transparent=False, bbox_inches='tight'); # for SPI overview